In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from collections import Counter
import pandas as pd
import random

SEED = 42
DATASET_NAME = "AICD-bench/AICD-Bench"

# T1 is the binary detection task.
# Its training set covers Python, Java and C++.
stream = load_dataset(
    DATASET_NAME,
    name="T1",
    split="train",
    streaming=True
)

print("Dataset stream created.")
print("Reading examples incrementally...")

samples = {0: [], 1: []}
counts = Counter()

for example in stream:
    code = example.get("code")
    label = example.get("label")

    if not isinstance(code, str) or not code.strip():
        continue

    if label not in (0, 1):
        continue

    counts[label] += 1

    if len(samples[label]) < 2500:
        samples[label].append({
            "code": code.strip(),
            "label": int(label)
        })

    if len(samples[0]) == 2500 and len(samples[1]) == 2500:
        break

print("Labels encountered:", counts)
print("Selected per label:", {k: len(v) for k, v in samples.items()})

if len(samples[0]) != 2500 or len(samples[1]) != 2500:
    raise ValueError(
        "Could not collect 2,500 samples for each label. "
        "Inspect the dataset's label values."
    )

data = pd.DataFrame(samples[0] + samples[1])
data = data.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("\nSelected dataset:", data.shape)
print(data["label"].value_counts())
display(data.head(3))

README.md:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

Dataset stream created.
Reading examples incrementally...
Labels encountered: Counter({1: 2887, 0: 2500})
Selected per label: {0: 2500, 1: 2500}

Selected dataset: (5000, 2)
label
0    2500
1    2500
Name: count, dtype: int64


,code,label
0,"def f(x, y, n):\n\treturn x + y if y == 0 or x...",0
1,word = input()\n print(word.capitalize())\n...,1
2,"class Solution:\n def threeSumMulti(self, a...",1


In [ ]:
train_df, temporary_df = train_test_split(
    data,
    test_size=0.30,
    random_state=SEED,
    stratify=data["label"]
)

val_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temporary_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTraining labels:")
print(train_df["label"].value_counts())

# Save the compact data so you do not need to stream it again.
train_df.to_csv("/content/aicd_train.csv", index=False)
val_df.to_csv("/content/aicd_validation.csv", index=False)
test_df.to_csv("/content/aicd_test.csv", index=False)

print("\nCompact CSV files saved in /content/")

Train: 3500
Validation: 750
Test: 750

Training labels:
label
1    1750
0    1750
Name: count, dtype: int64

Compact CSV files saved in /content/


In [ ]:
%pip install -q "datasets==3.6.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


In [ ]:
import datasets
import pandas as pd
import torch

print("Datasets:", datasets.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Datasets: 4.8.5
Pandas: 2.2.3
PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [ ]:
from datasets import get_dataset_config_names

DATASET_NAME = "AICD-bench/AICD-Bench"

try:
    configs = get_dataset_config_names(DATASET_NAME)
    print("Dataset is accessible!")
    print("Available configurations:", configs)

except Exception as e:
    print("Dataset access failed:")
    print(type(e).__name__, str(e)[:1500])

Dataset is accessible!
Available configurations: ['T1', 'T2', 'T3']


In [ ]:
from datasets import load_dataset

DATASET_NAME = "AICD-bench/AICD-Bench"

# Use T1, the binary classification task
stream = load_dataset(
    DATASET_NAME,
    name="T1",
    split="train",
    streaming=True
)

# Read only the first 5 examples
for i, example in enumerate(stream.take(5), start=1):
    print(f"\nExample {i}")
    print("Columns:", list(example.keys()))

    for key, value in example.items():
        if isinstance(value, str):
            print(f"{key}: {value[:300]}")
        else:
            print(f"{key}: {value}")


Example 1
Columns: ['code', 'label']
code: (a, b, c, d) = [int(x) for x in input().split()]
k = input()
(p, q, r, s) = (0, 0, 0, 0)
for i in k:
	if i == '1':
		p += 1
	elif i == '2':
		q += 1
	elif i == '3':
		r += 1
	elif i == '4':
		s += 1
print(a * p + b * q + c * r + d * s)

label: 0

Example 2
Columns: ['code', 'label']
code: valid version for the language; all others can also just parse a test that does an incorrect solution in real-time before testing all code solutions can cause the "wrong output".

The largest portion of this question lies in handling corner solutions without leading and trailing zeros.
Can a brute-f
label: 1

Example 3
Columns: ['code', 'label']
code: python
def min_cards_to_flip(s):
    vowels = {'a', 'e', 'i', 'o', 'u'}
    odd_digits = {'1', '3', '5', '7', '9'}

    # Collect all cards with vowels and all cards with odd digits
    cards_with_vowels = [i for i, char in enumerate(s) if char in vowels]
    cards_with_odd_digits = [i for i, char i
label: 1

E

In [ ]:
# Reuse the `stream` variable from Cell 4.
# Read a larger sample to inspect the schema and label format.

from collections import Counter

label_counts = Counter()
language_counts = Counter()
examples = []

for i, row in enumerate(stream):
    label = row.get("label")

    if label is not None:
        label_counts[str(label)] += 1

    language = row.get("language", row.get("lang", "unknown"))
    language_counts[str(language)] += 1

    if len(examples) < 3:
        examples.append(row)

    if i >= 999:
        break

print("Columns:", list(examples[0].keys()) if examples else "No rows")
print("\nLabel values in first 1,000 rows:", label_counts)
print("\nLanguage values:", language_counts)

for i, row in enumerate(examples, 1):
    print(f"\nExample {i}:")
    for key, value in row.items():
        print(f"{key}: {str(value)[:200]}")

Columns: ['code', 'label']

Label values in first 1,000 rows: Counter({'1': 538, '0': 462})

Language values: Counter({'unknown': 1000})

Example 1:
code: (a, b, c, d) = [int(x) for x in input().split()]
k = input()
(p, q, r, s) = (0, 0, 0, 0)
for i in k:
	if i == '1':
		p += 1
	elif i == '2':
		q += 1
	elif i == '3':
		r += 1
	elif i == '4':
		s += 1
p
label: 0

Example 2:
code: valid version for the language; all others can also just parse a test that does an incorrect solution in real-time before testing all code solutions can cause the "wrong output".

The largest portion 
label: 1

Example 3:
code: python
def min_cards_to_flip(s):
    vowels = {'a', 'e', 'i', 'o', 'u'}
    odd_digits = {'1', '3', '5', '7', '9'}

    # Collect all cards with vowels and all cards with odd digits
    cards_with_vow
label: 1


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    data,
    test_size=0.30,
    random_state=42,
    stratify=data["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

train_df.to_csv("/content/aicd_train.csv", index=False)
val_df.to_csv("/content/aicd_validation.csv", index=False)
test_df.to_csv("/content/aicd_test.csv", index=False)

Train: 3500
Validation: 750
Test: 750


In [ ]:
%pip install -q transformers peft accelerate bitsandbytes scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.7 MB/s eta 0:00:00


In [ ]:
import torch
import transformers
import peft
import accelerate
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
Transformers: 5.20.1
PEFT: 0.21.2
Accelerate: 1.15.0
BitsAndBytes: 0.50.2
CUDA available: True
GPU: Tesla T4


In [ ]:
import torch
import transformers
import peft
import accelerate
import bitsandbytes
import pandas as pd

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("Pandas:", pd.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise RuntimeError("Enable a GPU runtime in Colab before training.")

PyTorch: 2.11.0+cu130
Transformers: 5.20.1
PEFT: 0.21.2
Accelerate: 1.15.0
BitsAndBytes: 0.50.2
Pandas: 2.2.3
GPU available: True
GPU: Tesla T4
GPU memory (GB): 15.64


In [ ]:
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"

MAX_LENGTH = 512
EPOCHS = 2
LEARNING_RATE = 2e-4

PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

OUTPUT_DIR = "/content/deepseek-code-detector"
SEED = 42

print("Model:", MODEL_NAME)
print("Epochs:", EPOCHS)
print("Max sequence length:", MAX_LENGTH)
print("Effective batch size:",
      PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)

Model: deepseek-ai/deepseek-coder-1.3b-base
Epochs: 2
Max sequence length: 512
Effective batch size: 8


In [ ]:
import os
import pandas as pd

TRAIN_PATH = "/content/aicd_train.csv"
VAL_PATH = "/content/aicd_validation.csv"
TEST_PATH = "/content/aicd_test.csv"

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Cannot find {path}. Update the path to your saved CSV."
        )

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nColumns:", train_df.columns.tolist())
print("\nSample:")
display(train_df.head(2))

Train: (3500, 2)
Validation: (750, 2)
Test: (750, 2)

Columns: ['code', 'label']

Sample:


,code,label
0,for tt in range(t):\n n = int(input())\n ...,1
1,"from heapq import heappush, heappop \n\nclass ...",1


In [ ]:
def find_column(df, candidates):
    for name in candidates:
        if name in df.columns:
            return name
    return None

CODE_COL = find_column(
    train_df, ["code", "source_code", "content", "text"]
)

LABEL_COL = find_column(
    train_df, ["label", "target", "class", "generated"]
)

if CODE_COL is None or LABEL_COL is None:
    raise ValueError(
        f"Could not identify code/label columns. "
        f"Available columns: {train_df.columns.tolist()}"
    )

print("Code column:", CODE_COL)
print("Label column:", LABEL_COL)
print("\nOriginal label values:")
print(train_df[LABEL_COL].value_counts(dropna=False))

Code column: code
Label column: label

Original label values:
label
1    1750
0    1750
Name: count, dtype: int64


In [ ]:
def normalize_label(value):
    if pd.isna(value):
        return None

    value_str = str(value).strip().lower()

    if value_str in {"1", "1.0"}:
        return "AI"
    if value_str in {"0", "0.0"}:
        return "HUMAN"

    if "ai" in value_str or "generated" in value_str:
        return "AI"
    if "human" in value_str:
        return "HUMAN"

    raise ValueError(f"Unknown label: {value!r}")

for df in [train_df, val_df, test_df]:
    df["class_label"] = df[LABEL_COL].apply(normalize_label)
    df[CODE_COL] = df[CODE_COL].fillna("").astype(str)
    df.dropna(subset=["class_label"], inplace=True)
    df.drop_duplicates(subset=[CODE_COL, "class_label"], inplace=True)
    df.reset_index(drop=True, inplace=True)

print("Train labels:\n", train_df["class_label"].value_counts())
print("\nValidation labels:\n", val_df["class_label"].value_counts())
print("\nTest labels:\n", test_df["class_label"].value_counts())

assert set(train_df["class_label"]) == {"AI", "HUMAN"}, \
    "Training data must contain both classes."

Train labels:
 class_label
AI       1750
HUMAN    1750
Name: count, dtype: int64

Validation labels:
 class_label
AI       375
HUMAN    375
Name: count, dtype: int64

Test labels:
 class_label
AI       375
HUMAN    375
Name: count, dtype: int64


In [ ]:
print("Train labels:")
print(train_df[LABEL_COL].value_counts(dropna=False))

print("\nValidation labels:")
print(val_df[LABEL_COL].value_counts(dropna=False))

print("\nTest labels:")
print(test_df[LABEL_COL].value_counts(dropna=False))

print("\nColumns:", train_df.columns.tolist())

Train labels:
label
1    1750
0    1750
Name: count, dtype: int64

Validation labels:
label
1    375
0    375
Name: count, dtype: int64

Test labels:
label
1    375
0    375
Name: count, dtype: int64

Columns: ['code', 'label', 'class_label']


In [ ]:
print("Sample rows with labels:")
display(train_df[["code", "label"]].sample(6, random_state=42))

print("\nUnique label values:")
print(sorted(train_df["label"].dropna().unique()))

Sample rows with labels:


,code,label
1650,"n = int(input())\nlis = list(map(int, input()....",0
2456,works when the input is correct\n'''\n\n# capi...,1
2232,"class Solution:\n\n\tdef getCount(self, arr, n...",0
1945,"import math\ndef solve(a1, k):\n max_digit=...",1
309,import sys\nn = int(input().strip())\nscore = ...,0
2341,class Solution:\n def checkAlmostEquivalent...,0



Unique label values:
[np.int64(0), np.int64(1)]


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.config.use_cache = False

print("DeepSeek-Coder 1.3B loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/793 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.69GB            

... (2 outputs left out)

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

DeepSeek-Coder 1.3B loaded successfully!
GPU: Tesla T4


In [ ]:
from peft import (
    LoraConfig,
    TaskType,
    prepare_model_for_kbit_training,
    get_peft_model
)

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 3,145,728 || all params: 1,349,617,664 || trainable%: 0.2331


In [ ]:
from torch.utils.data import Dataset

MAX_LENGTH = 512

PROMPT_TEMPLATE = (
    "Classify the authorship of the following code as AI or HUMAN.\n"
    "Output only one label: AI or HUMAN.\n\n"
    "Code:\n"
)

class CodeClassificationDataset(Dataset):
    def __init__(self, dataframe):
        self.examples = []

        for _, row in dataframe.iterrows():
            code = str(row["code"]).strip()
            label = row["class_label"]

            if not code or label not in {"AI", "HUMAN"}:
                continue

            prompt_ids = tokenizer(
                PROMPT_TEMPLATE + code + "\n\nLabel:",
                add_special_tokens=True,
                truncation=True,
                max_length=MAX_LENGTH - 4
            )["input_ids"]

            target_ids = tokenizer(
                " " + label + tokenizer.eos_token,
                add_special_tokens=False
            )["input_ids"]

            input_ids = (prompt_ids + target_ids)[:MAX_LENGTH]
            labels = (
                [-100] * len(prompt_ids) + target_ids
            )[:MAX_LENGTH]

            self.examples.append({
                "input_ids": input_ids,
                "attention_mask": [1] * len(input_ids),
                "labels": labels
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


train_dataset = CodeClassificationDataset(train_df)
val_dataset = CodeClassificationDataset(val_df)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

Training examples: 3500
Validation examples: 750


In [ ]:
import torch

def data_collator(features):
    max_len = max(len(x["input_ids"]) for x in features)

    input_ids = []
    attention_mask = []
    labels = []

    for x in features:
        pad_len = max_len - len(x["input_ids"])

        input_ids.append(
            x["input_ids"] + [tokenizer.pad_token_id] * pad_len
        )
        attention_mask.append(
            x["attention_mask"] + [0] * pad_len
        )
        labels.append(
            x["labels"] + [-100] * pad_len
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long)
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/deepseek-code-detector",
    num_train_epochs=3,
    learning_rate=2e-4,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,

    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    seed=42,
    remove_unused_columns=False
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

train_result = trainer.train()

print("Training completed!")
print(train_result.metrics)

Epoch,Training Loss,Validation Loss
1,0.012088,0.026763
2,0.009780,0.029279
3,0.012833,0.028641


Training completed!
{'train_runtime': 4513.9115, 'train_samples_per_second': 2.326, 'train_steps_per_second': 0.291, 'total_flos': 2.2594658157563904e+16, 'train_loss': 0.07022754492626569, 'epoch': 3.0}


In [ ]:
import os
import zipfile
from google.colab import files

# Folder to save the trained model
MODEL_DIR = "/content/deepseek-code-detector/final_adapter"
ZIP_PATH = "/content/deepseek-code-detector.zip"

os.makedirs(MODEL_DIR, exist_ok=True)

# Save fine-tuned LoRA adapter and tokenizer
trainer.model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

# Create ZIP archive
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, filenames in os.walk(MODEL_DIR):
        for filename in filenames:
            filepath = os.path.join(root, filename)
            arcname = os.path.relpath(filepath, MODEL_DIR)
            zipf.write(filepath, arcname)

print("Model saved and zipped successfully!")
print("ZIP size:", round(os.path.getsize(ZIP_PATH) / (1024**2), 2), "MB")

# Download ZIP to your computer
files.download(ZIP_PATH)

NameError: name 'trainer' is not defined

In [ ]:
# Upgrade torchao to fix the incompatibility with PEFT
!pip install -q --upgrade torchao

import os
import shutil
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import files

BASE_MODEL = "deepseek-ai/deepseek-coder-1.3b-base"
ADAPTER_DIR = "/content/deepseek-code-detector/final_adapter"
MERGED_DIR = "/content/deepseek-code-detector/merged_model"
ZIP_PATH = "/content/deepseek-code-detector-full.zip"

print("Loading base model and adapter to merge...")
# Load base model in 16-bit float (not quantized, for merging)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu"
)

# Load the LoRA adapter on top of the base model
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

print("Merging model weights...")
# Merge adapter weights into the base model and unload LoRA
merged_model = peft_model.merge_and_unload()

print("Saving merged model and tokenizer...")
os.makedirs(MERGED_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_DIR)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print("Zipping merged model...")
shutil.make_archive(
    "/content/deepseek-code-detector-full",
    "zip",
    root_dir=MERGED_DIR
)

print(
    "Full model ZIP size:",
    round(os.path.getsize(ZIP_PATH) / (1024**3), 2),
    "GB"
)

print("Starting download...")
files.download(ZIP_PATH)

Loading base model and adapter to merge...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Merging model weights...
Saving merged model and tokenizer...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Zipping merged model...
Full model ZIP size: 2.05 GB
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments

# Define paths and configuration
MERGED_DIR = "/content/deepseek-code-detector/merged_model"
TEST_PATH = "/content/aicd_test.csv"

# 1. Re-load tokenizer and model if not already in memory
tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model for evaluation...")
model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 2. Re-create the Test Dataset
import pandas as pd
from torch.utils.data import Dataset

test_df = pd.read_csv(TEST_PATH)
test_df["class_label"] = test_df["label"].apply(lambda x: "AI" if str(x).strip() in {"1", "1.0"} else "HUMAN")

MAX_LENGTH = 512
PROMPT_TEMPLATE = (
    "Classify the authorship of the following code as AI or HUMAN.\n"
    "Output only one label: AI or HUMAN.\n\n"
    "Code:\n"
)

class CodeClassificationDataset(Dataset):
    def __init__(self, dataframe):
        self.examples = []
        for _, row in dataframe.iterrows():
            code = str(row["code"]).strip()
            label = row["class_label"]
            if not code or label not in {"AI", "HUMAN"}:
                continue
            prompt_ids = tokenizer(
                PROMPT_TEMPLATE + code + "\n\nLabel:",
                add_special_tokens=True,
                truncation=True,
                max_length=MAX_LENGTH - 4
            )["input_ids"]
            target_ids = tokenizer(
                " " + label + tokenizer.eos_token,
                add_special_tokens=False
            )["input_ids"]
            input_ids = (prompt_ids + target_ids)[:MAX_LENGTH]
            labels = ([-100] * len(prompt_ids) + target_ids)[:MAX_LENGTH]
            self.examples.append({
                "input_ids": input_ids,
                "attention_mask": [1] * len(input_ids),
                "labels": labels
            })
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        return self.examples[idx]

test_dataset = CodeClassificationDataset(test_df)

# 3. Re-define data collator
def data_collator(features):
    max_len = max(len(x["input_ids"]) for x in features)
    input_ids, attention_mask, labels = [], [], []
    for x in features:
        pad_len = max_len - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append(x["attention_mask"] + [0] * pad_len)
        labels.append(x["labels"] + [-100] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long)
    }

# 4. Initialize Trainer
training_args = TrainingArguments(
    output_dir="/content/temp_eval",
    per_device_eval_batch_size=1,
    fp16=True,
    remove_unused_columns=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator
)

# 5. Evaluate
print("Evaluating on test set...")
test_metrics = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test"
)

print("\nTest loss:", test_metrics.get("test_loss"))

Loading model for evaluation...


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Evaluating on test set...


Training Loss,Validation Loss,Step
No log,0.023442,0



Test loss: 0.02344169281423092


In [ ]:
import torch
import torch.nn.functional as F

AI_THRESHOLD = 0.60
HUMAN_THRESHOLD = 0.40

def detect_code_authorship(code):
    prompt = (
        "Classify the authorship of the following code as AI or HUMAN.\n"
        "Output only one label: AI or HUMAN.\n\n"
        f"Code:\n{code}\n\nLabel:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=504
    ).to(model.device)

    model.eval()

    def candidate_score(label):
        label_ids = tokenizer(
            " " + label,
            add_special_tokens=False
        )["input_ids"]

        context = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]
        total_score = 0.0

        with torch.no_grad():
            for token_id in label_ids:
                outputs = model(
                    input_ids=context,
                    attention_mask=attention_mask
                )

                next_logits = outputs.logits[0, -1, :]
                log_probs = F.log_softmax(next_logits.float(), dim=-1)

                total_score += log_probs[token_id].item()

                next_token = torch.tensor(
                    [[token_id]], device=context.device
                )
                context = torch.cat([context, next_token], dim=1)
                attention_mask = torch.cat(
                    [
                        attention_mask,
                        torch.ones(
                            (1, 1),
                            dtype=attention_mask.dtype,
                            device=attention_mask.device
                        )
                    ],
                    dim=1
                )

        return total_score

    ai_score = candidate_score("AI")
    human_score = candidate_score("HUMAN")

    scores = torch.tensor(
        [ai_score, human_score], dtype=torch.float32
    )
    relative = F.softmax(scores, dim=0)

    ai_confidence = relative[0].item()
    human_confidence = relative[1].item()

    if ai_confidence >= AI_THRESHOLD:
        prediction = "AI-generated"
    elif ai_confidence <= HUMAN_THRESHOLD:
        prediction = "Human-written"
    else:
        prediction = "Uncertain"

    return {
        "prediction": prediction,
        "ai_confidence": round(ai_confidence * 100, 2),
        "human_confidence": round(human_confidence * 100, 2)
    }

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# 1. Load independent dataset in streaming mode
dataset = load_dataset(
    "isThisYouLLM/H-AIRosettaMP",
    split="train",
    streaming=True
).shuffle(seed=42, buffer_size=10000)

# 2. Collect 200 AI + 200 human examples
samples = {"AI": [], "HUMAN": []}

for row in dataset:
    label = str(row["target"]).strip().lower()

    if label == "ai_generated" and len(samples["AI"]) < 200:
        samples["AI"].append(row)
    elif label == "human_written" and len(samples["HUMAN"]) < 200:
        samples["HUMAN"].append(row)

    if len(samples["AI"]) == 200 and len(samples["HUMAN"]) == 200:
        break

test_rows = samples["AI"] + samples["HUMAN"]

if not samples["AI"] or not samples["HUMAN"]:
    raise ValueError("Could not collect both classes from the dataset.")

print("Independent AI samples:", len(samples["AI"]))
print("Independent human samples:", len(samples["HUMAN"]))

# 3. Score candidate labels using the fine-tuned model
AI_THRESHOLD = 0.60
HUMAN_THRESHOLD = 0.40

model.eval()

def score_label(prompt, label):
    prompt_ids = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=504,
        add_special_tokens=True
    )["input_ids"].to(model.device)

    label_ids = tokenizer(
        " " + label,
        add_special_tokens=False
    )["input_ids"]

    target = torch.tensor(
        [label_ids], dtype=torch.long, device=model.device
    )
    full_ids = torch.cat([prompt_ids, target], dim=1)
    attention_mask = torch.ones_like(full_ids)

    with torch.no_grad():
        logits = model(
            input_ids=full_ids,
            attention_mask=attention_mask
        ).logits.float()

    start = prompt_ids.shape[1] - 1
    target_logits = logits[0, start:start + len(label_ids), :]
    log_probs = F.log_softmax(target_logits, dim=-1)

    token_scores = log_probs.gather(
        1, target[0].unsqueeze(1)
    ).squeeze(1)

    return token_scores.mean().item()


results = []

for row in tqdm(test_rows, desc="Testing unseen dataset"):
    code = str(row["code"])
    actual = (
        "AI" if str(row["target"]).lower() == "ai_generated"
        else "HUMAN"
    )

    prompt = (
        "Classify the authorship of this code as AI or HUMAN.\n"
        "Output only one label: AI or HUMAN.\n\n"
        f"Code:\n{code}\n\nLabel:"
    )

    ai_score = score_label(prompt, "AI")
    human_score = score_label(prompt, "HUMAN")

    probs = F.softmax(
        torch.tensor([ai_score, human_score]), dim=0
    )
    ai_conf = probs[0].item()
    human_conf = probs[1].item()

    if ai_conf >= AI_THRESHOLD:
        prediction = "AI"
    elif ai_conf <= HUMAN_THRESHOLD:
        prediction = "HUMAN"
    else:
        prediction = "UNCERTAIN"

    results.append({
        "language": row["language_name"],
        "actual": actual,
        "prediction": prediction,
        "ai_confidence": round(ai_conf * 100, 2),
        "human_confidence": round(human_conf * 100, 2)
    })

results_df = pd.DataFrame(results)

# 4. Overall metrics, using the higher-scoring label
actual = results_df["actual"]
predicted = [
    "AI" if a >= h else "HUMAN"
    for a, h in zip(
        results_df["ai_confidence"],
        results_df["human_confidence"]
    )
]

print("\n===== INDEPENDENT TEST RESULTS =====")
print("Samples:", len(results_df))
print("Accuracy:", round(accuracy_score(actual, predicted), 4))

print("\nClassification report:")
print(classification_report(
    actual, predicted,
    labels=["HUMAN", "AI"],
    zero_division=0
))

print("Confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(
    actual, predicted,
    labels=["HUMAN", "AI"]
))

# 5. Results after applying uncertainty thresholds
accepted = results_df[results_df["prediction"] != "UNCERTAIN"]

print("\nUncertain predictions:",
      (results_df["prediction"] == "UNCERTAIN").sum())
print("Coverage:", round(len(accepted) / len(results_df) * 100, 2), "%")

if len(accepted):
    print("Accuracy on accepted predictions:",
          round(accuracy_score(
              accepted["actual"], accepted["prediction"]
          ), 4))

# 6. Per-language results
results_df["binary_prediction"] = predicted

print("\n===== ACCURACY BY LANGUAGE =====")
for language, group in results_df.groupby("language"):
    print(
        language,
        "| samples:", len(group),
        "| accuracy:",
        round(accuracy_score(
            group["actual"], group["binary_prediction"]
        ), 4)
    )

results_df.to_csv(
    "/content/independent_code_test_results.csv",
    index=False
)
print("\nSaved results to /content/independent_code_test_results.csv")

Independent AI samples: 200
Independent human samples: 200


Testing unseen dataset:   0%|          | 0/400 [00:00<?, ?it/s]


===== INDEPENDENT TEST RESULTS =====
Samples: 400
Accuracy: 0.5325

Classification report:
              precision    recall  f1-score   support

       HUMAN       0.57      0.25      0.35       200
          AI       0.52      0.81      0.64       200

    accuracy                           0.53       400
   macro avg       0.55      0.53      0.49       400
weighted avg       0.55      0.53      0.49       400

Confusion matrix (rows=true, columns=predicted):
[[ 50 150]
 [ 37 163]]

Uncertain predictions: 86
Coverage: 78.5 %
Accuracy on accepted predictions: 0.5255

===== ACCURACY BY LANGUAGE =====
Java | samples: 400 | accuracy: 0.5325

Saved results to /content/independent_code_test_results.csv


In [ ]:
human_sample = """
def second_largest(nums):
    nums = list(set(nums))
    nums.sort()
    if len(nums) < 2:
        return None
    return nums[-2]
"""

ai_sample = """
from typing import List, Optional

def find_second_largest(numbers: List[int]) -> Optional[int]:
    unique_numbers = sorted(set(numbers), reverse=True)
    if len(unique_numbers) < 2:
        return None
    return unique_numbers[1]
"""

print("Sample 1:", detect_code_authorship(human_sample))
print("Sample 2:", detect_code_authorship(ai_sample))

Sample 1: {'prediction': 'AI-generated', 'ai_confidence': 99.56, 'human_confidence': 0.44}
Sample 2: {'prediction': 'AI-generated', 'ai_confidence': 99.55, 'human_confidence': 0.45}


In [ ]:
import torch
import torch.nn.functional as F

def score_candidate(code, label, use_mean=True):
    prompt = (
        "Classify the authorship of the following code as AI or HUMAN.\n"
        "Output only one label: AI or HUMAN.\n\n"
        f"Code:\n{code}\n\nLabel:"
    )

    prompt_ids = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=504,
        add_special_tokens=True
    )["input_ids"].to(model.device)

    label_ids = tokenizer(
        " " + label,
        add_special_tokens=False
    )["input_ids"]

    target = torch.tensor(
        [label_ids], dtype=torch.long, device=model.device
    )
    full_ids = torch.cat([prompt_ids, target], dim=1)
    attention_mask = torch.ones_like(full_ids)

    with torch.no_grad():
        logits = model(
            input_ids=full_ids,
            attention_mask=attention_mask
        ).logits.float()

    start = prompt_ids.shape[1] - 1
    target_logits = logits[0, start:start + len(label_ids), :]
    log_probs = F.log_softmax(target_logits, dim=-1)

    token_scores = log_probs.gather(
        1, target[0].unsqueeze(1)
    ).squeeze(1)

    if use_mean:
        return token_scores.mean().item()
    else:
        return token_scores.sum().item()

for name, code in [
    ("Sample 1", human_sample),
    ("Sample 2", ai_sample)
]:
    ai_mean = score_candidate(code, "AI", use_mean=True)
    human_mean = score_candidate(code, "HUMAN", use_mean=True)

    ai_total = score_candidate(code, "AI", use_mean=False)
    human_total = score_candidate(code, "HUMAN", use_mean=False)

    print(f"\n{name}")
    print(f"Mean scores:  AI={ai_mean:.4f}, HUMAN={human_mean:.4f}")
    print(f"Total scores: AI={ai_total:.4f}, HUMAN={human_total:.4f}")


Sample 1
Mean scores:  AI=-0.0048, HUMAN=-1.8094
Total scores: AI=-0.0048, HUMAN=-5.4282

Sample 2
Mean scores:  AI=-0.0049, HUMAN=-1.8042
Total scores: AI=-0.0049, HUMAN=-5.4126


In [ ]:
print("Model class:", type(model).__name__)
print("Model config:", model.config.model_type)
print("Architecture:", model.config.architectures)
print("Tokenizer vocab size:", len(tokenizer))
print("Model vocab size:", model.config.vocab_size)

Model class: LlamaForCausalLM
Model config: llama
Architecture: ['LlamaForCausalLM']
Tokenizer vocab size: 32022
Model vocab size: 32256


In [ ]:
import os
import json

ADAPTER_PATH = "/content/deepseek-code-detector/final_adapter"

config_path = os.path.join(ADAPTER_PATH, "adapter_config.json")

if os.path.exists(config_path):
    with open(config_path) as f:
        adapter_config = json.load(f)

    print(json.dumps(adapter_config, indent=2))
else:
    print("No adapter_config.json at:", config_path)

{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": null,
  "base_model_name_or_path": "deepseek-ai/deepseek-coder-1.3b-base",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "kasa_config": null,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 16,
  "lora_bias": false,
  "lora_dropout": 0.05,
  "lora_ga_config": null,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "monteclora_config": null,
  "peft_type": "LORA",
  "peft_version": "0.21.2",
  "qalora_group_size": 16,
  "r": 8,
  "rank_pattern": {},
  "revision": null,
  "target_modules": [
    "o_proj",
    "q_proj",
    "k_proj",
    "v_proj"
  ],
  "target_parameters": null,
  "task_type": "CAUSAL_LM",
  "trai

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for filename in files:
        if filename.endswith(".ipynb"):
            print(os.path.join(root, filename))

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Original held-out AICD-Bench test set
test_df = pd.read_csv("/content/aicd_test.csv")

# Use the label mapping from your notebook:
# 0 = HUMAN, 1 = AI
test_df["actual"] = test_df["label"].apply(
    lambda x: "AI" if int(x) == 1 else "HUMAN"
)

# Evaluate up to 200 samples
sample_df = test_df.sample(
    n=min(200, len(test_df)),
    random_state=42
)

predictions = []

for code in sample_df["code"].astype(str):
    ai_score = score_candidate(code, "AI", use_mean=True)
    human_score = score_candidate(code, "HUMAN", use_mean=True)

    predictions.append("AI" if ai_score > human_score else "HUMAN")

actual = sample_df["actual"].tolist()

print("AICD-Bench held-out test accuracy:",
      round(accuracy_score(actual, predictions), 4))

print("\nClassification report:")
print(classification_report(
    actual, predictions,
    labels=["HUMAN", "AI"],
    zero_division=0
))

print("Confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(
    actual, predictions,
    labels=["HUMAN", "AI"]
))

AICD-Bench held-out test accuracy: 0.97

Classification report:
              precision    recall  f1-score   support

       HUMAN       0.96      0.98      0.97       106
          AI       0.98      0.96      0.97        94

    accuracy                           0.97       200
   macro avg       0.97      0.97      0.97       200
weighted avg       0.97      0.97      0.97       200

Confusion matrix (rows=true, columns=predicted):
[[104   2]
 [  4  90]]


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load your previous independent test results
df = pd.read_csv("/content/independent_code_test_results.csv")

print("Columns:", df.columns.tolist())
print(df.head(2).to_string())

Columns: ['language', 'actual', 'prediction', 'ai_confidence', 'human_confidence', 'binary_prediction']
  language actual prediction  ai_confidence  human_confidence binary_prediction
0     Java     AI         AI          80.85             19.15                AI
1     Java     AI  UNCERTAIN          51.07             48.93                AI


In [ ]:
import os
import pandas as pd

for path in [
    "/content/aicd_test.csv",
    "/content/independent_code_test_results.csv"
]:
    if os.path.exists(path):
        df_check = pd.read_csv(path)
        print(f"\nFile: {path}")
        print("Rows:", len(df_check))
        print("Columns:", df_check.columns.tolist())


File: /content/aicd_test.csv
Rows: 750
Columns: ['code', 'label']

File: /content/independent_code_test_results.csv
Rows: 400
Columns: ['language', 'actual', 'prediction', 'ai_confidence', 'human_confidence', 'binary_prediction']


In [ ]:
from datasets import load_dataset
import pandas as pd

# Recreate the same balanced independent sample
stream = load_dataset(
    "isThisYouLLM/H-AIRosettaMP",
    split="train",
    streaming=True
).shuffle(seed=42, buffer_size=10000)

ai_samples = []
human_samples = []

for row in stream:
    target = row["target"]

    if target == "Ai_generated" and len(ai_samples) < 200:
        ai_samples.append({
            "code": row["code"],
            "actual": "AI",
            "language": row["language_name"]
        })

    elif target == "Human_written" and len(human_samples) < 200:
        human_samples.append({
            "code": row["code"],
            "actual": "HUMAN",
            "language": row["language_name"]
        })

    if len(ai_samples) == 200 and len(human_samples) == 200:
        break

independent_df = pd.DataFrame(ai_samples + human_samples)

print("Samples:", len(independent_df))
print(independent_df["actual"].value_counts())
print(independent_df["language"].value_counts())

Samples: 400
actual
AI       200
HUMAN    200
Name: count, dtype: int64
language
Java    400
Name: count, dtype: int64


In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

def evaluate_method(use_mean):
    predictions = []

    for i, row in enumerate(independent_df.itertuples(), start=1):
        ai_score = score_candidate(row.code, "AI", use_mean=use_mean)
        human_score = score_candidate(row.code, "HUMAN", use_mean=use_mean)

        predictions.append("AI" if ai_score > human_score else "HUMAN")

        if i % 50 == 0:
            print(f"{'Mean' if use_mean else 'Total'} scoring: {i}/400")

    actual = independent_df["actual"].tolist()

    print("\nMethod:", "MEAN log-probability" if use_mean else "TOTAL log-probability")
    print("Accuracy:", round(accuracy_score(actual, predictions), 4))
    print(classification_report(
        actual, predictions,
        labels=["HUMAN", "AI"],
        zero_division=0
    ))
    print("Confusion matrix (rows=true, columns=predicted):")
    print(confusion_matrix(actual, predictions, labels=["HUMAN", "AI"]))

    return predictions

mean_predictions = evaluate_method(use_mean=True)
total_predictions = evaluate_method(use_mean=False)

independent_df["mean_prediction"] = mean_predictions
independent_df["total_prediction"] = total_predictions
independent_df.to_csv(
    "/content/scoring_comparison.csv",
    index=False
)

print("\nSaved comparison to /content/scoring_comparison.csv")

Mean scoring: 50/400
Mean scoring: 100/400
Mean scoring: 150/400
Mean scoring: 200/400
Mean scoring: 250/400
Mean scoring: 300/400
Mean scoring: 350/400
Mean scoring: 400/400

Method: MEAN log-probability
Accuracy: 0.5225
              precision    recall  f1-score   support

       HUMAN       0.54      0.28      0.37       200
          AI       0.52      0.77      0.62       200

    accuracy                           0.52       400
   macro avg       0.53      0.52      0.49       400
weighted avg       0.53      0.52      0.49       400

Confusion matrix (rows=true, columns=predicted):
[[ 55 145]
 [ 46 154]]
Total scoring: 50/400
Total scoring: 100/400
Total scoring: 150/400
Total scoring: 200/400
Total scoring: 250/400
Total scoring: 300/400
Total scoring: 350/400
Total scoring: 400/400

Method: TOTAL log-probability
Accuracy: 0.4875
              precision    recall  f1-score   support

       HUMAN       0.46      0.14      0.22       200
          AI       0.49      0.83      

In [ ]:
import pandas as pd

df = pd.read_csv("/content/scoring_comparison.csv")

for col in ["mean_prediction", "total_prediction"]:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))
    print("\nPrediction by actual class:")
    print(pd.crosstab(df["actual"], df[col]))


mean_prediction
mean_prediction
AI       299
HUMAN    101
Name: count, dtype: int64

Prediction by actual class:
mean_prediction   AI  HUMAN
actual                     
AI               154     46
HUMAN            145     55

total_prediction
total_prediction
AI       337
HUMAN     63
Name: count, dtype: int64

Prediction by actual class:
total_prediction   AI  HUMAN
actual                      
AI                166     34
HUMAN             171     29
